# Ejercicio 4 — Transformer Encoder-Decoder para traducción automática

En este ejercicio vamos a construir una arquitectura Transformer completa formada por:

- **Encoder**: procesa la frase de entrada en inglés.
- **Decoder**: genera la frase de salida en español token a token.

El objetivo es traducir frases muy simples de inglés a español.

Este ejercicio es una prueba de concepto, por lo que el dataset es muy pequeño. No buscamos un traductor real, sino entender cómo se conectan encoder y decoder dentro de un Transformer.

## 1. Importación de librerías

Usaremos TensorFlow/Keras para construir el modelo Transformer, y las herramientas de tokenización de Keras para convertir palabras en identificadores numéricos.

In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import (
    Input,
    Embedding,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    Dropout
)

from tensorflow.keras.models import Model

# Semillas para reproducibilidad aproximada
np.random.seed(42)
tf.random.set_seed(42)

## 2. Dataset de frases

El enunciado proporciona tres frases en inglés y sus traducciones al español.

Aunque el conjunto es muy pequeño, es suficiente para ver cómo se entrena un modelo encoder-decoder básico.

In [2]:
sentences_english = [
    "hello world",
    "goodbye world",
    "hello everyone"
]

sentences_spanish = [
    "hola mundo",
    "adiós mundo",
    "hola a todos"
]

for en, es in zip(sentences_english, sentences_spanish):
    print(f"{en}  ->  {es}")

hello world  ->  hola mundo
goodbye world  ->  adiós mundo
hello everyone  ->  hola a todos


## 3. Añadir tokens especiales `<bos>` y `<eos>`

En el decoder necesitamos indicar explícitamente:

- `<bos>`: *begin of sequence*, comienzo de la frase que se va a generar.
- `<eos>`: *end of sequence*, final de la frase generada.

Durante la traducción, el decoder empieza con `<bos>` y va generando palabras hasta producir `<eos>`.

In [3]:
sentences_spanish = [
    f"<bos> {sent} <eos>"
    for sent in sentences_spanish
]

print(sentences_spanish)

['<bos> hola mundo <eos>', '<bos> adiós mundo <eos>', '<bos> hola a todos <eos>']


## 4. Tokenización del encoder

El encoder recibe las frases en inglés.

Primero creamos un tokenizer para el inglés, después convertimos las frases en secuencias de números y finalmente aplicamos padding.

Ejemplo conceptual:

```text
"hello world" -> [2, 3]
```

El modelo no trabaja con palabras directamente, sino con identificadores numéricos.

In [4]:
encoder_tokenizer = Tokenizer(filters='', oov_token='<OOV>')
encoder_tokenizer.fit_on_texts(sentences_english)

encoder_seq = encoder_tokenizer.texts_to_sequences(sentences_english)

encoder_vocab_size = len(encoder_tokenizer.word_index) + 1
encoder_max_len = max(len(seq) for seq in encoder_seq)

encoder_inputs = pad_sequences(
    encoder_seq,
    maxlen=encoder_max_len,
    padding='post'
)

print("Vocabulario encoder:")
print(encoder_tokenizer.word_index)

print("Secuencias encoder:")
print(encoder_seq)

print("Inputs encoder con padding:")
print(encoder_inputs)

print("encoder_vocab_size:", encoder_vocab_size)
print("encoder_max_len:", encoder_max_len)

Vocabulario encoder:
{'<OOV>': 1, 'hello': 2, 'world': 3, 'goodbye': 4, 'everyone': 5}
Secuencias encoder:
[[2, 3], [4, 3], [2, 5]]
Inputs encoder con padding:
[[2 3]
 [4 3]
 [2 5]]
encoder_vocab_size: 6
encoder_max_len: 2


## 5. Tokenización del decoder

El decoder trabaja con las frases en español.

Para entrenarlo, creamos dos conjuntos:

- `decoder_inputs`: frase sin el último token.
- `decoder_targets`: frase sin el primer token.

Ejemplo:

```text
Frase completa:  <bos> hola mundo <eos>
Entrada decoder: <bos> hola mundo
Salida esperada: hola mundo <eos>
```

Así el modelo aprende a predecir el siguiente token en español.

In [6]:
decoder_tokenizer = Tokenizer(filters='', oov_token='<OOV>')
decoder_tokenizer.fit_on_texts(sentences_spanish)

decoder_seq = decoder_tokenizer.texts_to_sequences(sentences_spanish)

decoder_vocab_size = len(decoder_tokenizer.word_index) + 1
decoder_max_len = max(len(seq) for seq in decoder_seq)

# Entrada del decoder: secuencia sin el último token
decoder_input_seq = [seq[:-1] for seq in decoder_seq]

# Target del decoder: secuencia sin el primer token
decoder_target_seq = [seq[1:] for seq in decoder_seq]

decoder_inputs = pad_sequences(
    decoder_input_seq,
    maxlen=decoder_max_len - 1,
    padding='post'
)

decoder_targets = pad_sequences(
    decoder_target_seq,
    maxlen=decoder_max_len - 1,
    padding='post'
)

print("Vocabulario decoder:")
print(decoder_tokenizer.word_index)

print("Secuencias decoder completas:")
print(decoder_seq)

print("Inputs decoder:")
print(decoder_inputs)

print("Targets decoder:")
print(decoder_targets)

print("decoder_vocab_size:", decoder_vocab_size)
print("decoder_max_len:", decoder_max_len)

Vocabulario decoder:
{'<OOV>': 1, '<bos>': 2, '<eos>': 3, 'hola': 4, 'mundo': 5, 'adiós': 6, 'a': 7, 'todos': 8}
Secuencias decoder completas:
[[2, 4, 5, 3], [2, 6, 5, 3], [2, 4, 7, 8, 3]]
Inputs decoder:
[[2 4 5 0]
 [2 6 5 0]
 [2 4 7 8]]
Targets decoder:
[[4 5 3 0]
 [6 5 3 0]
 [4 7 8 3]]
decoder_vocab_size: 9
decoder_max_len: 5


## 6. Parámetros del modelo

Definimos algunos hiperparámetros sencillos:

- `embedding_dim`: tamaño de los vectores de embedding.
- `num_heads`: número de cabezas de atención.
- `ff_dim`: dimensión de la capa feed-forward interna.
- `dropout`: regularización.

Como el dataset es muy pequeño, usamos valores bajos para mantener el modelo simple.

In [7]:
embedding_dim = 32
num_heads = 2
ff_dim = 64
dropout = 0.1

## 7. Bloque Transformer Encoder

El encoder recibe la frase en inglés y genera una representación contextual de esa frase.

Su estructura básica es:

1. Self-attention.
2. Conexión residual + normalización.
3. Red feed-forward.
4. Nueva conexión residual + normalización.

El encoder no genera tokens directamente. Su función es **entender la entrada**.

In [10]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    # Self-attention del encoder
    attn = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=head_size
    )(
        query=inputs,
        value=inputs,
        key=inputs
    )

    attn = Dropout(dropout)(attn)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attn)

    # Feed-forward
    ff = Dense(ff_dim, activation='relu')(out1)
    ff = Dense(inputs.shape[-1])(ff)
    ff = Dropout(dropout)(ff)

    return LayerNormalization(epsilon=1e-6)(out1 + ff)

## 8. Bloque Transformer Decoder

El decoder tiene dos atenciones principales:

### 1. Masked self-attention

El decoder mira los tokens que ya ha generado, pero no puede mirar tokens futuros.

Esto se consigue con:

```python
use_causal_mask=True
```

### 2. Cross-attention

El decoder consulta la salida del encoder para generar la traducción correcta.

In [8]:
def transformer_decoder(inputs, encoder_output, head_size, num_heads, ff_dim, dropout=0.1):
    # 1. Masked self-attention del decoder
    attn1 = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=head_size
    )(
        query=inputs,
        value=inputs,
        key=inputs,
        use_causal_mask=True
    )

    attn1 = Dropout(dropout)(attn1)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attn1)

    # 2. Cross-attention: el decoder atiende a la salida del encoder
    attn2 = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=head_size
    )(
        query=out1,
        value=encoder_output,
        key=encoder_output
    )

    attn2 = Dropout(dropout)(attn2)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + attn2)

    # Feed-forward
    ff = Dense(ff_dim, activation='relu')(out2)
    ff = Dense(out2.shape[-1])(ff)
    ff = Dropout(dropout)(ff)

    return LayerNormalization(epsilon=1e-6)(out2 + ff)

## 9. Construcción del modelo encoder-decoder

Ahora conectamos todo:

1. Entrada del encoder: frase en inglés.
2. Entrada del decoder: frase parcial en español.
3. El encoder procesa el inglés.
4. El decoder usa su entrada parcial y la salida del encoder.
5. La capa final predice el siguiente token en español para cada posición.

El modelo recibe dos entradas:

```python
Model([encoder_input, decoder_input], outputs)
```

In [11]:
# Entrada del encoder
en_in = Input(shape=(encoder_max_len,), dtype=tf.int32, name='encoder_input')

# Entrada del decoder
dec_in = Input(shape=(decoder_max_len - 1,), dtype=tf.int32, name='decoder_input')

# Embeddings del encoder
encoder_embedding = Embedding(
    input_dim=encoder_vocab_size,
    output_dim=embedding_dim,
    name='encoder_embedding'
)(en_in)

# Bloque encoder
encoder_output = transformer_encoder(
    encoder_embedding,
    head_size=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim,
    dropout=dropout
)

# Embeddings del decoder
decoder_embedding = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=embedding_dim,
    name='decoder_embedding'
)(dec_in)

# Bloque decoder
decoder_output = transformer_decoder(
    decoder_embedding,
    encoder_output,
    head_size=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim,
    dropout=dropout
)

# Predicción del vocabulario español en cada posición
outputs = Dense(decoder_vocab_size, name='output_vocab')(decoder_output)

# Modelo completo
model = Model([en_in, dec_in], outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 2, 32)     │        192 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 2, 32)     │      8,416 │ encoder_embeddin… │
│ (MultiHeadAttentio… │                   │            │ encoder_embeddin… │
│                     │                   │            │ encoder_embeddin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 2, 32)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 2, 32)     │          0 │ encoder_embeddin… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 2, 32)     │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 4, 32)     │        288 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 2, 64)     │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 4, 32)     │      8,416 │ decoder_embeddin… │
│ (MultiHeadAttentio… │                   │            │ decoder_embeddin… │
│                     │                   │            │ decoder_embeddin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 2, 32)     │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 4, 32)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 2, 32)     │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 4, 32)     │          0 │ decoder_embeddin… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 2, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 4, 32)     │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 2, 32)     │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 4, 32)     │      8,416 │ layer_normalizat

 Total params: 34,729 (135.66 KB)

 Trainable params: 34,729 (135.66 KB)

 Non-trainable params: 0 (0.00 B)

## 10. Compilación del modelo

Usamos `SparseCategoricalCrossentropy(from_logits=True)` porque:

- Las etiquetas son enteros, no vectores one-hot.
- La última capa `Dense(decoder_vocab_size)` no tiene `softmax`.

El modelo aprende a predecir un token del vocabulario español en cada posición.

In [12]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

## 11. Entrenamiento

Entrenamos el modelo pasando dos entradas:

```python
[encoder_inputs, decoder_inputs]
```

Y como salida esperada:

```python
decoder_targets
```

El enunciado propone `epochs=5`, pero con solo tres frases el modelo suele necesitar más épocas para memorizar el pequeño conjunto. Por eso dejamos `epochs=300` para que se pueda observar la traducción correctamente.

Si quieres ajustarte estrictamente al enunciado, cambia `epochs=300` por `epochs=5`.

In [13]:
history = model.fit(
    [encoder_inputs, decoder_inputs],
    decoder_targets,
    epochs=300,
    batch_size=1,
    verbose=1
)

Epoch 1/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0833 - loss: 2.9805  
Epoch 2/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4167 - loss: 1.9696
Epoch 3/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6667 - loss: 1.5991 
Epoch 4/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6667 - loss: 1.2492 
Epoch 5/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6667 - loss: 1.0558 
Epoch 6/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7500 - loss: 0.8751
Epoch 7/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8333 - loss: 0.7609
Epoch 8/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8333 - loss: 0.5989
Epoch 9/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9167 - loss: 0.4577 
Epoch 10/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.3502
Epoch 11/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.2608 
Epoch 12/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - l

## 12. Función de traducción

La traducción se realiza de forma autoregresiva:

1. Se tokeniza la frase en inglés.
2. Se empieza el decoder con `<bos>`.
3. El modelo predice el siguiente token.
4. Ese token se añade a la entrada del decoder.
5. Se repite hasta generar `<eos>` o alcanzar la longitud máxima.

Este proceso es muy parecido a la generación del Ejercicio 3, pero ahora el decoder está condicionado por la salida del encoder.

In [14]:
def translate_sentence(input_sentence):
    eos_token = decoder_tokenizer.word_index['<eos>']
    bos_token = decoder_tokenizer.word_index['<bos>']

    # Tokenizar frase de entrada en inglés
    encoder_input = encoder_tokenizer.texts_to_sequences([input_sentence])
    encoder_input = pad_sequences(
        encoder_input,
        maxlen=encoder_max_len,
        padding='post'
    )

    # El decoder empieza con <bos>
    decoder_output = [bos_token]

    # Generación token a token
    for _ in range(decoder_max_len - 1):
        decoder_input = pad_sequences(
            [decoder_output],
            maxlen=decoder_max_len - 1,
            padding='post'
        )

        pred = model.predict([encoder_input, decoder_input], verbose=0)

        # Elegimos el token más probable en la posición actual
        next_token = np.argmax(pred[0, len(decoder_output) - 1])

        decoder_output.append(next_token)

        if next_token == eos_token:
            break

    # Convertimos IDs a palabras y quitamos tokens especiales
    output_tokens = [
        decoder_tokenizer.index_word[token]
        for token in decoder_output
        if token != eos_token and token != bos_token and token in decoder_tokenizer.index_word
    ]

    return ' '.join(output_tokens)

## 13. Prueba de traducción

Probamos las frases del conjunto de entrenamiento.

Como solo hay tres frases, lo normal es que el modelo tienda a memorizarlas.

In [15]:
for sentence in sentences_english:
    translation = translate_sentence(sentence)
    print(f"{sentence}  ->  {translation}")

hello world  ->  hola mundo
goodbye world  ->  adiós mundo
hello everyone  ->  hola a todos


## 14. Prueba adicional

Podemos probar con una frase nueva, aunque el modelo probablemente no generalice bien debido al tamaño mínimo del dataset.

Esto es normal: un modelo Transformer necesita muchísimos más ejemplos para aprender traducción real.

In [16]:
print("Traducción de 'hello world':", translate_sentence("hello world"))
print("Traducción de 'goodbye world':", translate_sentence("goodbye world"))
print("Traducción de 'hello everyone':", translate_sentence("hello everyone"))

# Frase nueva, fuera del dataset
print("Traducción de 'goodbye everyone':", translate_sentence("goodbye everyone"))

Traducción de 'hello world': hola mundo
Traducción de 'goodbye world': adiós mundo
Traducción de 'hello everyone': hola a todos
Traducción de 'goodbye everyone': adiós a todos


## 15. Conclusión

En este ejercicio hemos construido una arquitectura Transformer completa con encoder y decoder.

El **encoder** procesa la frase en inglés y genera una representación contextual de entrada.

El **decoder** genera la traducción en español token a token, usando:

- **Masked self-attention**, para no mirar tokens futuros.
- **Cross-attention**, para consultar la información generada por el encoder.

La salida final del modelo es una distribución sobre el vocabulario español en cada posición.

La arquitectura encoder-decoder se usa cuando queremos transformar una secuencia de entrada en otra secuencia de salida, como en traducción automática, resumen de texto o corrección gramatical.

En este caso, el dataset es muy pequeño, por lo que el modelo tiende a memorizar. Aun así, sirve para comprender la estructura básica de un Transformer de traducción.